<a href="https://colab.research.google.com/github/chohyerinn/DataScience_HotelBooking/blob/main/Hotel_Booking_Cancellation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STEP 1 to 5. Refine data and verify basic logistic regression

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================
# STEP 1. Load Dataset
# ==============================

df = pd.read_csv("hotel_bookings.csv")

print(df.head())
print(df.shape)
print(df.info())
print(df.describe())
print(df.isnull().sum())
print(df.duplicated().sum())
print(df["is_canceled"].value_counts())


# ==============================
# STEP 2. Feature Engineering
# ==============================

# Create total guests
df["total_guests"] = df["adults"] + df["children"] + df["babies"]

# Create total stays
df["total_stays"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]

# Check engineered features
print(df[["total_guests", "total_stays"]].head())

# Check invalid records
print((df["total_guests"] == 0).sum())
print((df["total_stays"] == 0).sum())
print((df["adr"] < 0).sum())

print(df.shape)

df_no_dup = df.drop_duplicates()
print(df_no_dup.shape)


# ==============================
# STEP 3. Data Cleaning
# ==============================

# Fill missing values
df["children"] = df["children"].fillna(0)
df["country"] = df["country"].fillna("Unknown")
df["agent"] = df["agent"].fillna(0)
df["company"] = df["company"].fillna(0)

# Remove invalid records
df = df[df["total_guests"] > 0]
df = df[df["total_stays"] > 0]
df = df[df["adr"] >= 0]

# Remove duplicate rows
df = df.drop_duplicates()

# Check final cleaned dataset
print(df.shape)
print(df.isnull().sum())

# Save cleaned dataset
df.to_csv("hotel_bookings_cleaned.csv", index=False)


# ==============================
# STEP 4. EDA Visualization
# ==============================

sns.set_style("whitegrid")

# Visualization 1: Cancellation Distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="is_canceled")
plt.title("Booking Cancellation Distribution")
plt.xlabel("Canceled")
plt.ylabel("Count")
plt.savefig("booking_cancellation_distribution.png", bbox_inches="tight")
plt.close()

# Visualization 2: Hotel Type vs Cancellation
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="hotel", hue="is_canceled")
plt.title("Hotel Type vs Cancellation")
plt.xlabel("Hotel Type")
plt.ylabel("Count")
plt.savefig("hotel_type_vs_cancellation.png", bbox_inches="tight")
plt.close()

# Visualization 3: Lead Time Distribution
plt.figure(figsize=(8, 5))
sns.histplot(df["lead_time"], bins=50)
plt.title("Lead Time Distribution")
plt.xlabel("Lead Time")
plt.ylabel("Frequency")
plt.savefig("lead_time_distribution.png", bbox_inches="tight")
plt.close()

# Visualization 4: ADR Distribution
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, y="adr")
plt.title("ADR Distribution")
plt.savefig("adr_distribution.png", bbox_inches="tight")
plt.close()

# Visualization 5: Deposit Type vs Cancellation
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="deposit_type", hue="is_canceled")
plt.title("Deposit Type vs Cancellation")
plt.xlabel("Deposit Type")
plt.ylabel("Count")
plt.savefig("deposit_type_vs_cancellation.png", bbox_inches="tight")
plt.close()


# ==============================
# STEP 5. Feature Selection & Encoding
# ==============================

# Remove leakage columns
df = df.drop([
    "reservation_status",
    "reservation_status_date"
], axis=1)

# Remove unnecessary identifier columns
df = df.drop([
    "agent",
    "company"
], axis=1)

# Check remaining columns
print(df.columns)

# Convert categorical variables using one-hot encoding
df_encoded = pd.get_dummies(df, drop_first=True)

# Check encoded dataset
print(df_encoded.head())
print(df_encoded.shape)


# ==============================
# STEP 5. Train-Test Split
# ==============================

from sklearn.model_selection import train_test_split

# Features
X = df_encoded.drop("is_canceled", axis=1)

# Target
y = df_encoded["is_canceled"]

print(X.shape)
print(y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


# ==============================
# STEP 5. Logistic Regression Model
# ==============================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# Create Logistic Regression model
log_model = LogisticRegression(max_iter=1000)

# Train model
log_model.fit(X_train, y_train)

# Predict test data
y_pred_log = log_model.predict(X_test)


# ==============================
# STEP 5. Logistic Regression Evaluation
# ==============================

print("Logistic Regression Accuracy:")
print(accuracy_score(y_test, y_pred_log))

print("Logistic Regression Precision:")
print(precision_score(y_test, y_pred_log))

print("Logistic Regression Recall:")
print(recall_score(y_test, y_pred_log))

print("Logistic Regression F1 Score:")
print(f1_score(y_test, y_pred_log))

print("Logistic Regression Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_log))

print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_log))


# ==============================
# STEP 5. Save Logistic Regression Confusion Matrix Image
# ==============================

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_log,
    display_labels=["Not Canceled", "Canceled"]
)

plt.title("Logistic Regression Confusion Matrix")
plt.savefig("logistic_confusion_matrix.png", bbox_inches="tight")
plt.close()

          hotel  is_canceled  lead_time  arrival_date_year arrival_date_month  \
0  Resort Hotel            0        342               2015               July   
1  Resort Hotel            0        737               2015               July   
2  Resort Hotel            0          7               2015               July   
3  Resort Hotel            0         13               2015               July   
4  Resort Hotel            0         14               2015               July   

   arrival_date_week_number  arrival_date_day_of_month  \
0                        27                          1   
1                        27                          1   
2                        27                          1   
3                        27                          1   
4                        27                          1   

   stays_in_weekend_nights  stays_in_week_nights  adults  ...  deposit_type  \
0                        0                     0       2  ...    No Deposit   
1     

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression Accuracy:
0.7842673284469326
Logistic Regression Precision:
0.6708474576271186
Logistic Regression Recall:
0.4169827222924568
Logistic Regression F1 Score:
0.5142931392931392
Logistic Regression Confusion Matrix:
[[11610   971]
 [ 2767  1979]]
Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.92      0.86     12581
           1       0.67      0.42      0.51      4746

    accuracy                           0.78     17327
   macro avg       0.74      0.67      0.69     17327
weighted avg       0.77      0.78      0.77     17327



# STEP 6. 5-Fold Cross Validation

In [11]:
# ==============================================================================
# STEP 6. Data Scaling & K-Fold Cross Validation
# ==============================================================================
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

print("\n" + "="*50 + "\n STEP 6. Data Scaling & K-Fold Cross Validation\n" + "="*50)

# List of continuous variables
numerical_cols = ['lead_time', 'adr', 'stays_in_weekend_nights', 'stays_in_week_nights',
                  'adults', 'children', 'babies', 'booking_changes', 'total_of_special_requests',
                  'total_guests', 'total_stays']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Perform data scaling
scaler = StandardScaler()
X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

# Create a Default Model Object
models_to_test = {
    "Scaled Logistic Regression": LogisticRegression(max_iter=500),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier()
}

# Accumulated calculation structure of detailed evaluation indicators through KFold Declaration and for Loop
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Slicing top 15,000 samples to prevent infinite browser loading due to computational resource limitations
X_cv = X_train_scaled.iloc[:15000].reset_index(drop=True)
y_cv = y_train.iloc[:15000].reset_index(drop=True)

print("\n--- 5-Fold Cross Validation Results (Averaged using kf.split) ---")
for name, model in models_to_test.items():
    accuracy_list = []
    precision_list = []
    recall_list = []
    f1_list = []

    for train_idx, val_idx in kf.split(X_cv):
        X_tr, X_val = X_cv.iloc[train_idx], X_cv.iloc[val_idx]
        y_tr, y_val = y_cv.iloc[train_idx], y_cv.iloc[val_idx]

        # Learning and Predicting
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        # Calculate evaluation indicators and record them in the list
        accuracy_list.append(accuracy_score(y_val, preds))
        precision_list.append(precision_score(y_val, preds))
        recall_list.append(recall_score(y_val, preds))
        f1_list.append(f1_score(y_val, preds))

    print(f"[{name}]")
    print(f"   CV Accuracy : {np.mean(accuracy_list):.4f}")
    print(f"   CV Precision: {np.mean(precision_list):.4f}")
    print(f"   CV Recall   : {np.mean(recall_list):.4f}")
    print(f"   CV F1-Score : {np.mean(f1_list):.4f}\n")


 STEP 6. Data Scaling & K-Fold Cross Validation

--- 5-Fold Cross Validation Results (Averaged using kf.split) ---
[Scaled Logistic Regression]
   CV Accuracy : 0.7781
   CV Precision: 0.6472
   CV Recall   : 0.4107
   CV F1-Score : 0.5020

[Decision Tree]
   CV Accuracy : 0.7652
   CV Precision: 0.5682
   CV Recall   : 0.5794
   CV F1-Score : 0.5736

[K-Nearest Neighbors]
   CV Accuracy : 0.7205
   CV Precision: 0.4829
   CV Recall   : 0.3475
   CV F1-Score : 0.4039



# STEP 7. Unsupervised Learning K-means Cluster Analysis and Elbow Method

In [12]:
# ==============================================================================
# STEP 7. K-means Clustering Analysis
# ==============================================================================
from sklearn.cluster import KMeans

print("\n" + "="*50 + "\nSTEP 7. K-means Clustering Analysis\n" + "="*50)

# 1. Select key numerical features for clustering
cluster_features = ['lead_time', 'adr', 'total_guests', 'total_stays', 'booking_changes', 'total_of_special_requests']

# To secure computational speed and avoid indexing errors, index initialization after clearly slicing only the corresponding numerical variables in scaled training data
X_cluster_scaled = X_train_scaled[cluster_features].reset_index(drop=True)

# 2. Optimum k navigation with Elbow Method
# After performing K-Means for various k candidate groups,
# Record sum of squared errors (Inertia) between points and center points in the cluster
sse_list = []
k_range = range(1, 7)

# Explore with top 5,000 samples to prevent environmental infinite waiting due to excessive bulk data operations
X_elbow_sample = X_cluster_scaled.iloc[:5000]

for k in k_range:
    # Declare a pure default instance with the hardware parallel option removed
    kmeans_ideal = KMeans(n_clusters=k, random_state=42)
    kmeans_ideal.fit(X_elbow_sample)
    sse_list.append(kmeans_ideal.inertia_)

# Creating and saving Elbow Method Visualization Graphs
plt.figure(figsize=(7, 4))
plt.plot(k_range, sse_list, marker='o', color='b')
plt.title("Elbow Method for Optimal k (Week13 Standard)")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Sum of Squared Errors (SSE)")
plt.grid(True)
plt.savefig("kmeans_elbow_method.png", bbox_inches="tight")
plt.close()
print("[System] The Elbow Method analysis graph was saved as 'kmeans_elbow_method.png' file.")


# 3. Learn three core customer groups (k=3) definitions and standard models
kmeans_final = KMeans(n_clusters=3, random_state=42)
kmeans_final.fit(X_cluster_scaled) # Use labels_extraction after fit

# Standard mapping of cluster assignment labels to data frames after completion of training
X_cluster_scaled['cluster'] = kmeans_final.labels_


# 4. Generating an average statistics summary table by cluster for interpretation of a business perspective (no loss of content)
# Copy the iloc reference from the source df to ensure that the index is as secure as the size of the train set index to prevent distortion bugs
df_interpretation = df.loc[X_train.index].copy().reset_index(drop=True)
df_interpretation['cluster'] = X_cluster_scaled['cluster']

# Aggregate average statistics and actual cancellation rates (is_canceled) of numerical variables for each group (Cluster)
cluster_summary = df_interpretation.groupby('cluster')[cluster_features + ['is_canceled']].mean()
cluster_summary['booking_count'] = df_interpretation.groupby('cluster').size()

print("\n[Business Interpreting Table] Summary of Average Characteristics by Cluster (Restore the original data scale)")
print(cluster_summary.round(2).to_string())


# 5. Creating and storing visualization data
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_interpretation.sample(n=2000, random_state=42), # Visualize 2,000 samples for chart readability
    x='lead_time',
    y='adr',
    hue='cluster',
    palette='Set1',
    alpha=0.7
)
plt.title("Hotel Booking Segments (K-means k=3)")
plt.xlabel("Lead Time (Days)")
plt.ylabel("Average Daily Rate (ADR)")
plt.savefig("hotel_clusters_visualization.png", bbox_inches="tight")
plt.close()
print("\n[System] Cluster scatterplot visualization graph 'hotel_clusters_visualization.png' is saved.")


STEP 7. K-means Clustering Analysis
[System] The Elbow Method analysis graph was saved as 'kmeans_elbow_method.png' file.

[Business Interpreting Table] Summary of Average Characteristics by Cluster (Restore the original data scale)
         lead_time     adr  total_guests  total_stays  booking_changes  total_of_special_requests  is_canceled  booking_count
cluster                                                                                                                      
0           194.66   96.48          2.01         6.10             0.43                       0.60         0.37          15847
1            36.27   85.43          1.72         2.61             0.19                       0.40         0.23          34999
2            65.56  157.69          2.64         3.50             0.27                       1.35         0.29          18461

[System] Cluster scatterplot visualization graph 'hotel_clusters_visualization.png' is saved.


# STEP 8. Open Source Contribution Specifications Integrated Top-Level Pipeline Function

In [13]:
# ==============================================================================
# STEP 8. Open Source SW Contribution (Top-Level Function)
# ==============================================================================
# We built a pipeline engine with a loop scheme (kf.split).
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier


print("\n" + "="*50 + "\nSTEP 8. Open Source SW Function & Top 5\n" + "="*50)

def fit_and_evaluate_hotel_pipeline(cleaned_df, numerical_features, target_col="is_canceled", k_folds=5):
    """
    Integrated pipeline function for open-source contributions in Pandas and Scikit-learn styles.
    The standard control loop traverses the preprocessing scheme and algorithmic options to return the optimal top five combinations.
    """
    # 1. Copy and feature encoding to prevent damage to the original data frame (Step 5 method is maintained)
    df_temp = cleaned_df.copy()
    X_features = df_temp.drop(target_col, axis=1)
    y_target = df_temp[target_col]
    X_encoded = pd.get_dummies(X_features, drop_first=True)

    # 2. Perform even sampling (Holdout standard) to prevent large computational resource limitations (infinite loading and server down)
    # Use the strict option to keep the original target distribution ratio secure.
    X_sample, _, y_sample, _ = train_test_split(
        X_encoded, y_target, train_size=3000, random_state=42, stratify=y_target
    )

    # Initialize indexes for data mapping reliability
    X_sample = X_sample.reset_index(drop=True)
    y_sample = y_sample.reset_index(drop=True)

    # 3. Define a variety of data scaling methods
    scalers_dict = {
        'Standard': StandardScaler(),
        'MinMax': MinMaxScaler()
    }

    # 4. Define hyperparameter experimental table
    # Configuring the maximum depth max_depth combination of regulatory strength C of logistic regression and decision tree
    log_reg_C_list = [0.1, 1.0]
    dec_tree_depth_list = [5, 10]

    results_list = []
    kf_pipe = KFold(n_splits=k_folds, shuffle=True, random_state=42)

    # 5. Manual deployment of all optimal combinations to pure triple-overlap for loops without a surrogate function (cross_validate)
    for scaler_name, scaler_obj in scalers_dict.items():
        # Independent copy of data by scaler and transform continuous variables
        X_scaled_base = X_sample.copy()
        X_scaled_base[numerical_features] = scaler_obj.fit_transform(X_sample[numerical_features])

        # Logistic Regression Hyperparameter Combination Validation Circuit
        for c_val in log_reg_C_list:
            f1_scores = []

            # Entering KFold Indexing Segmentation
            for train_idx, val_idx in kf_pipe.split(X_scaled_base):
                X_tr, X_val = X_scaled_base.iloc[train_idx], X_scaled_base.iloc[val_idx]
                y_tr, y_val = y_sample.iloc[train_idx], y_sample.iloc[val_idx]

                model = LogisticRegression(C=c_val, max_iter=500)
                model.fit(X_tr, y_tr)
                preds = model.predict(X_val)
                f1_scores.append(f1_score(y_val, preds))

            results_list.append({
                'Scaler': scaler_name,
                'Algorithm': 'LogisticRegression',
                'Hyperparameters': f"C={c_val}",
                'Mean_CV_F1_Score': round(np.mean(f1_scores), 4)
            })

        # Decision Tree Hyperparameter Combination Validation Circuit
        for depth_val in dec_tree_depth_list:
            f1_scores = []

            # Entering KFold Indexing Segmentation
            for train_idx, val_idx in kf_pipe.split(X_scaled_base):
                X_tr, X_val = X_scaled_base.iloc[train_idx], X_scaled_base.iloc[val_idx]
                y_tr, y_val = y_sample.iloc[train_idx], y_sample.iloc[val_idx]

                model = DecisionTreeClassifier(max_depth=depth_val, random_state=42)
                model.fit(X_tr, y_tr)
                preds = model.predict(X_val)
                f1_scores.append(f1_score(y_val, preds))

            results_list.append({
                'Scaler': scaler_name,
                'Algorithm': 'DecisionTree',
                'Hyperparameters': f"max_depth={depth_val}",
                'Mean_CV_F1_Score': round(np.mean(f1_scores), 4)
            })

    # 6. Pre-processing+parameter experimental table results are sorted in order of performance and extracted from the top 5
    results_df = pd.DataFrame(results_list)
    top_5_combinations = results_df.sort_values(by='Mean_CV_F1_Score', ascending=False).head(5)

    return top_5_combinations

# Integrated open source module operation test and best combination report table output
print("Executing Automated Pipeline for Open Source Contribution Specification...")
top_5_report = fit_and_evaluate_hotel_pipeline(df, numerical_cols, target_col="is_canceled", k_folds=5)

print("\nTOP 5 BEST COMBINATIONS")
print(top_5_report.to_string(index=False))


STEP 8. Open Source SW Function & Top 5
Executing Automated Pipeline for Open Source Contribution Specification...

TOP 5 BEST COMBINATIONS
  Scaler          Algorithm Hyperparameters  Mean_CV_F1_Score
  MinMax       DecisionTree     max_depth=5            0.5836
Standard       DecisionTree     max_depth=5            0.5826
Standard       DecisionTree    max_depth=10            0.5363
  MinMax       DecisionTree    max_depth=10            0.5349
Standard LogisticRegression           C=1.0            0.4935
